In [1]:
# external
import numpy as np, healpy as hp, time, ducc0
from tqdm import trange
# modules from cmblensplus
import cmblensplus.curvedsky as cs
from cmblensplus.utils import constant as c, cmb

In [2]:
def benchmark(func, niter, desc):
    times = []

    for _ in trange(niter, desc=desc):
        t0 = time.perf_counter()
        func()
        times.append(time.perf_counter() - t0)

    times = np.asarray(times)

    print(f"{desc}")
    print(f"median: {np.median(times)}")
    print(f"mean:   {np.mean(times)}")
    print(f"min:    {np.min(times)}")


In [3]:
niter = 10

In [4]:
Lmax = 2*2048
# ucl is an array of shape [0:5,0:rlmax+1] and ucl[0,:] = TT, ucl[1,:] = EE, ucl[2,:] = TE, lcl[3,:] = phiphi, lcl[4,:] = Tphi
ucl = cmb.read_camb_cls('../data/unlensedcls.dat',ftype='scal',output='array')[:,:Lmax+1]

Generate random gaussian fields

In [5]:
Talm = cs.utils.gauss1alm(ucl[0,:])

In [6]:
alm = hp.synalm(ucl[0,:], lmax=Lmax, new=True)

Set parameters

In [7]:
nside = 1024  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside

In [8]:
Tmap = cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1])

In [9]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [10]:
benchmark(
    lambda: cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1]),
    niter=niter,
    desc="cs hp_alm2map",
)

cs hp_alm2map: 100%|██████████| 10/10 [00:08<00:00,  1.17it/s]

cs hp_alm2map
median: 0.8495973385870457
mean:   0.850737042305991
min:    0.8335125311277807


In [11]:
alm_ducc = alm.reshape(1, -1)
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.synthesis(alm=alm_ducc,map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 synthesis",
)

ducc0 synthesis: 100%|██████████| 10/10 [00:07<00:00,  1.33it/s]

ducc0 synthesis
median: 0.7507313704118133
mean:   0.753365934244357
min:    0.7407104189042002


In [12]:
benchmark(
    lambda: cs.utils.hp_map2alm(lmax,lmax,Tmap),
    niter=niter,
    desc="cs hp_map2alm",
)

cs hp_map2alm: 100%|██████████| 10/10 [00:09<00:00,  1.10it/s]

cs hp_map2alm
median: 0.9018526633735746
mean:   0.9073310817591846
min:    0.8869481249712408


In [13]:
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.adjoint_synthesis(map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 adjoint_synthesis",
)

ducc0 adjoint_synthesis: 100%|██████████| 10/10 [00:07<00:00,  1.27it/s]

ducc0 adjoint_synthesis
median: 0.7841991309542209
mean:   0.7885213532717898
min:    0.7633937441278249


Higher resolution

In [14]:
nside = 2048  # Nside of Healpix map
npix  = 12*nside**2  # Numer of pixels of Healpix map
lmax  = 2*nside # Maximum multipole of harmonic coefficients to be computed

In [15]:
Tmap = cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1])

In [16]:
hb = ducc0.healpix.Healpix_Base(nside, "RING")
mp = np.empty((1, hb.npix()), dtype=np.float64)

In [17]:
benchmark(
    lambda: cs.utils.hp_alm2map(nside,Talm[:lmax+1, :lmax+1]),
    niter=niter,
    desc="cs hp_alm2map",
)

cs hp_alm2map: 100%|██████████| 10/10 [00:53<00:00,  5.38s/it]

cs hp_alm2map
median: 5.368035982945003
mean:   5.382585147977807
min:    5.340583293000236


In [18]:
alm_ducc = alm.reshape(1, -1)
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.synthesis(alm=alm_ducc,map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 synthesis",
)

ducc0 synthesis: 100%|██████████| 10/10 [00:52<00:00,  5.20s/it]

ducc0 synthesis
median: 5.180715391878039
mean:   5.199242226965725
min:    5.1183962929062545


In [19]:
benchmark(
    lambda: cs.utils.hp_map2alm(lmax,lmax,Tmap),
    niter=niter,
    desc="cs hp_map2alm",
)

cs hp_map2alm: 100%|██████████| 10/10 [00:59<00:00,  5.93s/it]

cs hp_map2alm
median: 5.930383444530889
mean:   5.9253118696855385
min:    5.875690787099302


In [20]:
sht_info = hb.sht_info()
benchmark(
    lambda: ducc0.sht.adjoint_synthesis(map=mp,spin=0,lmax=lmax,mmax=lmax,nthreads=2,**sht_info),
    niter=niter,
    desc="ducc0 adjoint_synthesis",
)

ducc0 adjoint_synthesis: 100%|██████████| 10/10 [00:54<00:00,  5.47s/it]

ducc0 adjoint_synthesis
median: 5.460234135505743
mean:   5.469026210322045
min:    5.392376654082909
